In [54]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import sys
import os

# Get the absolute path to the directory containing the current script
# For GlobalLocal/src/analysis/preproc/make_epoched_data.py, this is GlobalLocal/src/analysis/preproc
# Get the absolute path to the directory containing the current script
try:
    # This will work if running as a .py script
    current_file_path = os.path.abspath(__file__)
    current_script_dir = os.path.dirname(current_file_path)
except NameError:
    # This will be executed if __file__ is not defined (e.g., in a Jupyter Notebook)
    current_script_dir = os.getcwd()

# Navigate up two levels to get to the 'GlobalLocal' directory
project_root = os.path.abspath(os.path.join(current_script_dir, '..', '..'))

# Add the 'GlobalLocal' directory to sys.path if it's not already there
if project_root not in sys.path:
    sys.path.insert(0, project_root)  # insert at the beginning to prioritize it

import numpy as np
import json
import mne
results_dir = '/cwork/jz421/BIDS-1.1_GlobalLocal/BIDS/derivatives/freqFilt/figs/Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLength_0.5s_stat_func_ttest_ind_equal_var_False_nan_policy_omit/saved_results'
save_dir = "/hpc/home/jz421/coganlab/jz421/GlobalLocal/dcc_scripts/power/cns26_poster_plots"
os.listdir(results_dir)
from src.analysis.config.plotting_parameters import plotting_parameters as plot_params
from src.analysis.power.power_traces import plot_power_traces_for_all_rois


In [55]:
metadata = {}
metadata['congruency'] = json.load(open(os.path.join(results_dir, 'stimulus_congruency_conditions_17_subjects_metadata.json')))
metadata['switch_type'] = json.load(open(os.path.join(results_dir, 'stimulus_switch_type_conditions_17_subjects_metadata.json')))
first_effect_name = next(iter(metadata.keys()))
rois = metadata[first_effect_name]['rois']

In [56]:
metadata['congruency']

{'condition_names': ['Stimulus_i', 'Stimulus_c'],
 'rois': ['lpfc'],
 'conditions_save_name': 'stimulus_congruency_conditions_17_subjects',
 'sfreq': 256}

In [57]:
# Rebuild evks_dict_elecs

evks_dict_elecs = {}
significant_clusters = {}
n_subjects = 17

for effect_name, effect_metadata in metadata.items():
    evks_dict_elecs[effect_name] = {}
    for cond in effect_metadata['condition_names']:
        evks_dict_elecs[effect_name][cond] = {}
        for roi in effect_metadata['rois']:
            f = np.load(f'{results_dir}/stimulus_{effect_name}_conditions_{n_subjects}_subjects_{cond}_{roi}_evoked.npz', allow_pickle=True)
            # Reconstruct as MNE Evoked
            info = mne.create_info(list(f['ch_names']), sfreq=effect_metadata['sfreq'], ch_types='seeg')
            evks_dict_elecs[effect_name][cond][roi] = mne.EvokedArray(f['data'], info, tmin=f['times'][0])
    
    # Load clusters
    clusters_file = np.load(f'{results_dir}/stimulus_{effect_name}_conditions_{n_subjects}_subjects_significant_clusters.npz')
    significant_clusters[effect_name] = {roi: clusters_file[roi] for roi in effect_metadata['rois']}

In [ ]:
# plotting
PLOT_STYLE = {
    # Toggles
    'show_title': False,
    'show_xlabel': False,
    'show_ylabel': False,
    'show_legend': False,
    
    # Labels
    'title': None,        # None = auto-generate from ROI name
    'x_label': 'Time (s)',
    'y_label': 'Power (z)',
    
    # Font sizes
    'title_font_size': 14,
    'axis_font_size': 12,
    'tick_font_size': 12,
    'legend_font_size': 10,
    
    # Tick customization
    'xticks': [-1.0, -0.5, 0, 0.5, 1.0, 1.5],       # None = auto, or pass array
    'yticks': [-0.1, 0, 0.1, 0.2, 0.3],
    'xtick_labels': None, # Custom labels for xticks (like 'baseline', 'onset', etc), which are placed at the xtick locations.
    'ytick_labels': None,
    'xlim': (-1, 1.5),
    'ylim': (-0.1, 0.3),
    
    # Other
    'figsize': (12, 8),
    'text_color': 'black',
    'sig_cluster_height': 0.3,
}

In [59]:
for effect_name, effect_name_evks_dict_elecs in evks_dict_elecs.items():
    plot_power_traces_for_all_rois(
        effect_name_evks_dict_elecs, rois, metadata[effect_name]['condition_names'], metadata[effect_name]['conditions_save_name'], plot_params,
        window_size=window_size, sampling_rate=metadata[effect_name]['sfreq'], 
        significant_clusters=significant_clusters[effect_name],
        save_dir=save_dir,
        error_type='sem',
        plot_style=PLOT_STYLE,
        save_name_suffix=f'{effect_name}_power_traces',
)

Saved plot to: /hpc/home/jz421/coganlab/jz421/GlobalLocal/dcc_scripts/power/cns26_poster_plots/lpfc_stimulus_congruency_conditions_17_subjects_congruency_power_traces_sem_shading.png

All plots saved to: /hpc/home/jz421/coganlab/jz421/GlobalLocal/dcc_scripts/power/cns26_poster_plots
Saved plot to: /hpc/home/jz421/coganlab/jz421/GlobalLocal/dcc_scripts/power/cns26_poster_plots/lpfc_stimulus_switch_type_conditions_17_subjects_switch_type_power_traces_sem_shading.png

All plots saved to: /hpc/home/jz421/coganlab/jz421/GlobalLocal/dcc_scripts/power/cns26_poster_plots
